<a href="https://colab.research.google.com/github/SinnottKayleigh/B2B-Sales-Algos/blob/main/Proximal_Policy_Optimization_(PPO).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Proximal Policy Optimisation (PPO) belongs to the family of policy gradients.
- Designed to improve the stability and reliabillity of training, preffered for thise with continuous action spaces and complex environments.
- The PPO policy is updaed iteratively based on the gradient of the reward signal.

**Clipping Mechanism:**

- Restricts how much the policy can change in a single update.
- Helps prevent large updates that can destabilise training.
- The objective function for PPO involves a clipped version of the ratio between the new and old policies. By clipping this ratio into a small interval, PPO ensures that the policy does not change too drastically, which helps maintain stability.
- The PPO objective is a "surrogate objective" meaning its an approximation of the true objective (expected return).
- The surrogate objective balances the trade off between exploration and exploitation.
- Uses an "advantage function" to measure how much better a particular action is compared to the average behaviour under the current policy. This helps to focus learning on the most beneficial actions. Common methods used includ the generalised advantage estomation (GAE) which combines bootstrapped value estimates and Temporal Difference errors to improve the bias-variance trade off.
- An "on-policy" algorithm, thus it updates the policy based on data collected from the current policy. In contrast, DQN is an off-policy algorithm, which learns data collected from any policy.

In [15]:
import numpy as np
import pandas as pd
from faker import Faker
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Normal
from sklearn.preprocessing import StandardScaler

# Seed for reproducibility
np.random.seed(42)
torch.manual_seed(42)

class B2BSaaSDataGenerator:
    def __init__(self, num_samples=1000):
        """
        Generate synthetic B2B SaaS sales dataset

        Args:
            num_samples (int): Number of synthetic data points to generate
        """
        self.faker = Faker()
        self.num_samples = num_samples

    def generate_dataset(self):
        """
        Generate synthetic B2B SaaS sales dataset

        Returns:
            pd.DataFrame: Synthetic sales dataset
        """
        data = {
            'company_id': [self.faker.unique.uuid4() for _ in range(self.num_samples)],
            'company_size': np.random.choice(
                ['startup', 'small', 'medium', 'enterprise'],
                size=self.num_samples,
                p=[0.3, 0.3, 0.2, 0.2]
            ),
            'industry': np.random.choice(
                ['tech', 'finance', 'healthcare', 'retail', 'education'],
                size=self.num_samples
            ),
            'annual_revenue': np.random.lognormal(mean=10, sigma=1, size=self.num_samples),
            'funding_stage': np.random.choice(
                ['seed', 'series_a', 'series_b', 'series_c', 'public'],
                size=self.num_samples
            ),
            'current_software_budget': np.random.uniform(1000, 100000, size=self.num_samples),
            'software_complexity_score': np.random.uniform(1, 10, size=self.num_samples),
            'previous_saas_purchases': np.random.randint(0, 10, size=self.num_samples),
            'sales_cycle_length_days': np.random.normal(loc=45, scale=15, size=self.num_samples),
            'conversion_probability': np.random.uniform(0, 1, size=self.num_samples)
        }

        # Create DataFrame
        df = pd.DataFrame(data)

        # Add some feature engineering
        df['budget_to_revenue_ratio'] = df['current_software_budget'] / df['annual_revenue']
        df['is_high_potential_lead'] = (
            (df['company_size'].isin(['enterprise', 'medium'])) &
            (df['conversion_probability'] > 0.7)
        ).astype(int)

        return df

class CustomPPOAgent:
    """
    Custom Proximal Policy Optimization (PPO) Algorithm Implementation
    """
    def __init__(self, state_dim, action_dim, lr=1e-3, clip_ratio=0.2):
        """
        Initialize PPO Agent

        Args:
            state_dim (int): Dimension of input state
            action_dim (int): Dimension of action space
            lr (float): Learning rate
            clip_ratio (float): Clipping parameter for policy updates
        """
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # Policy Network (Actor)
        self.actor = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_dim)
        ).to(self.device)

        # Value Network (Critic)
        self.critic = nn.Sequential(
            nn.Linear(state_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        ).to(self.device)

        # Learnable log standard deviation
        self.log_std = nn.Parameter(torch.zeros(action_dim, device=self.device))

        # Optimizers
        self.actor_optimizer = optim.Adam(
            list(self.actor.parameters()) + [self.log_std],
            lr=lr
        )
        self.critic_optimizer = optim.Adam(self.critic.parameters(), lr=lr)

        # Hyperparameters
        self.clip_ratio = clip_ratio
        self.state_dim = state_dim
        self.action_dim = action_dim

    def select_action(self, state):
        """
        Select action using current policy

        Args:
            state (np.array): Current state

        Returns:
            action (np.array): Selected action
            log_prob (float): Log probability of the action
        """
        # Ensure state is a torch tensor on the correct device
        state = torch.FloatTensor(state).to(self.device)

        # Get mean from actor network
        mean = self.actor(state)

        # Compute standard deviation
        std = torch.exp(self.log_std)

        # Create normal distribution
        dist = Normal(mean, std)

        # Sample action
        action = dist.rsample()

        # Compute log probability
        log_prob = dist.log_prob(action).sum()

        return action.cpu().detach().numpy(), log_prob.item()

    def update(self, states, actions, old_log_probs, returns, advantages):
        """
        PPO update step

        Args:
            states (np.array): States
            actions (np.array): Actions taken
            old_log_probs (np.array): Log probabilities of old actions
            returns (np.array): Computed returns
            advantages (np.array): Computed advantages
        """
        # Convert to tensors on the correct device
        states = torch.FloatTensor(states).to(self.device)
        actions = torch.FloatTensor(actions).to(self.device)
        old_log_probs = torch.FloatTensor(old_log_probs).to(self.device)
        returns = torch.FloatTensor(returns).to(self.device)
        advantages = torch.FloatTensor(advantages).to(self.device)

        # Normalize advantages
        if advantages.numel() > 0:
            advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)

        # Compute current policy distribution
        means = self.actor(states)
        std = torch.exp(self.log_std)
        dist = Normal(means, std)

        # Compute log probabilities of current actions
        log_probs = dist.log_prob(actions).sum(dim=1)

        # Compute policy loss (PPO update)
        ratios = torch.exp(log_probs - old_log_probs)
        surr1 = ratios * advantages
        surr2 = torch.clamp(ratios, 1.0 - self.clip_ratio, 1.0 + self.clip_ratio) * advantages
        policy_loss = -torch.min(surr1, surr2).mean()

        # Compute value loss
        values = self.critic(states).squeeze()
        value_loss = F.mse_loss(values, returns.squeeze())

        # Total loss
        loss = policy_loss + 0.5 * value_loss

        # Update networks
        self.actor_optimizer.zero_grad()
        self.critic_optimizer.zero_grad()
        loss.backward()
        self.actor_optimizer.step()
        self.critic_optimizer.step()

        return loss.item()

    def get_value(self, state):
        """
        Get value of a state

        Args:
            state (np.array): Input state

        Returns:
            value (float): Estimated value of the state
        """
        state = torch.FloatTensor(state).to(self.device)
        return self.critic(state).cpu().item()

class StatePreprocessor:
    """
    Preprocessor for converting raw data to numeric features
    """
    def __init__(self, dataset):
        """
        Initialize state preprocessor

        Args:
            dataset (pd.DataFrame): Input dataset
        """
        # Categorical mapping
        self.categorical_mapping = {
            'company_size': {'startup': 0, 'small': 1, 'medium': 2, 'enterprise': 3},
            'industry': {'tech': 0, 'finance': 1, 'healthcare': 2, 'retail': 3, 'education': 4},
            'funding_stage': {'seed': 0, 'series_a': 1, 'series_b': 2, 'series_c': 3, 'public': 4}
        }

        # Prepare categorical columns
        self.categorical_cols = list(self.categorical_mapping.keys())

        # Numeric columns to use
        self.numeric_cols = [
            'annual_revenue',
            'current_software_budget',
            'software_complexity_score',
            'previous_saas_purchases',
            'sales_cycle_length_days',
            'conversion_probability',
            'budget_to_revenue_ratio'
        ]

        # All feature columns
        self.all_feature_cols = self.categorical_cols + self.numeric_cols

        # Scaler for numeric features
        self.scaler = StandardScaler()

        # Fit the scaler
        self.scaler.fit(dataset[self.numeric_cols].values)

    def preprocess_state(self, state):
        """
        Preprocess state to convert to numeric features

        Args:
            state (pd.Series or np.array): Input state

        Returns:
            np.array: Numeric features for PPO agent
        """
        # Prepare feature vector
        features = []

        # Process categorical columns
        for col in self.categorical_cols:
            # Map categorical variables
            features.append(self.categorical_mapping[col].get(state[col], -1))

        # Process numeric columns
        numeric_features = state[self.numeric_cols].values
        numeric_scaled = self.scaler.transform([numeric_features])[0]
        features.extend(numeric_scaled)

        return np.array(features, dtype=np.float32)

def main():
    # Generate B2B SaaS dataset
    data_generator = B2BSaaSDataGenerator(num_samples=1000)
    dataset = data_generator.generate_dataset()

    # Save dataset
    dataset.to_csv('b2b_saas_synthetic_dataset.csv', index=False)
    print("Synthetic B2B SaaS dataset generated and saved.")

    # Create state preprocessor
    preprocessor = StatePreprocessor(dataset)

    # Example of setting up PPO Agent
    state_dim = len(preprocessor.all_feature_cols)  # Number of numeric features
    action_dim = 2  # Example: sales strategy adjustment

    # Initialize PPO Agent
    ppo_agent = CustomPPOAgent(state_dim, action_dim)

    # Simulated training loop
    num_episodes = 100
    for episode in range(num_episodes):
        # Simulate state, action, reward collection
        raw_state = dataset.sample(1).iloc[0]

        # Preprocess state to numeric features
        state = preprocessor.preprocess_state(raw_state)

        # Select action
        action, log_prob = ppo_agent.select_action(state)

        # Placeholder reward and value computation
        # Could be based on conversion probability, budget, etc.
        reward = raw_state['conversion_probability'] * raw_state['current_software_budget'] / 10000
        value = ppo_agent.get_value(state)

        # Update agent (simplified)
        loss = ppo_agent.update(
            states=[state],
            actions=[action],
            old_log_probs=[log_prob],
            returns=[reward],
            advantages=[reward - value]
        )

        if episode % 10 == 0:
            print(f"Episode {episode}: Reward = {reward:.4f}, Loss = {loss:.4f}")

if __name__ == "__main__":
    main()

Synthetic B2B SaaS dataset generated and saved.
Episode 0: Reward = 8.1259, Loss = nan


<ipython-input-15-7dc9f89a45cf>:168: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)


ValueError: Expected parameter loc (Tensor of shape (2,)) of distribution Normal(loc: torch.Size([2]), scale: torch.Size([2])) to satisfy the constraint Real(), but found invalid values:
tensor([nan, nan], grad_fn=<ViewBackward0>)

In [2]:
pip install faker

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 21.7 MB/s eta 0:00:00



**Hybrid Approach - Supervised Learning with Reinforcement Learning**



In [7]:
def advanced_feature_engineering(self, data):
    """
    Enhanced feature creation for more nuanced insights
    """
    # Temporal features
    data['days_since_last_interaction'] = (
        current_date - data['last_interaction_date']
    ).dt.days

    # Interaction complexity score
    data['interaction_complexity'] = (
        data['num_touchpoints'] * data['communication_depth']
    )

    # Predictive engagement indicators
    data['engagement_score'] = (
        0.4 * data['email_open_rate'] +
        0.3 * data['meeting_scheduled_rate'] +
        0.3 * data['content_download_rate']
    )

    # Propensity to convert based on multiple signals
    data['conversion_propensity'] = logistic_regression_model.predict_proba(
        data[key_predictive_features]
    )

In [8]:
class HierarchicalSalesAI:
    def __init__(self):
        # Initial supervised learning phase
        self.base_model = SupervisedBaseModel()

        # Adaptive reinforcement learning layer
        self.adaptive_rl_layer = AdaptiveRLAgent()

        # Ensemble meta-learner
        self.meta_learner = EnsembleCombiner()

    def predict(self, new_data):
        # Multi-stage prediction
        base_predictions = self.base_model.predict(new_data)
        rl_refinements = self.adaptive_rl_layer.refine(new_data)
        final_predictions = self.meta_learner.combine(
            base_predictions,
            rl_refinements
        )
        return final_predictions

Dynamic Reward Mechanism:

In [9]:
def create_advanced_reward_function(sales_interaction):
    """
    Multidimensional reward calculation
    """
    reward_components = {
        'revenue_potential': calculate_revenue_potential(sales_interaction),
        'deal_progression': measure_pipeline_movement(sales_interaction),
        'customer_fit': assess_ideal_customer_profile_match(sales_interaction),
        'sales_efficiency': compute_resource_utilization(sales_interaction)
    }

    # Weighted reward calculation
    total_reward = (
        0.4 * reward_components['revenue_potential'] +
        0.3 * reward_components['deal_progression'] +
        0.2 * reward_components['customer_fit'] +
        0.1 * reward_components['sales_efficiency']
    )

    return total_reward

In [10]:
def quantify_prediction_uncertainty(predictions):
    """
    Measure confidence and uncertainty in predictions
    """
    prediction_variance = np.var(predictions)
    prediction_entropy = calculate_entropy(predictions)

    uncertainty_metrics = {
        'variance': prediction_variance,
        'entropy': prediction_entropy,
        'confidence_interval': compute_confidence_interval(predictions)
    }

    return uncertainty_metrics

In [11]:
class ContinuousLearningPipeline:
    def __init__(self):
        self.model_registry = ModelVersionRegistry()
        self.performance_tracker = ModelPerformanceTracker()

    def update_model(self, new_data, model_performance):
        # Automated model selection and update
        if model_performance.is_significant_drift():
            new_model_version = train_new_model(new_data)
            self.model_registry.register(new_model_version)
            self.performance_tracker.log_transition()
